# Escalated case: the ladder again, on the orders that reached customers

**Week 1, Tuesday afternoon. The escalated case, unguided, fifty minutes.** The morning climbed six
chapters on booked orders. This case climbs the same ladder alone, on a harder definition, where one
branch that held in the morning moves.

> **The client asks.** "The board pack reports revenue on orders that reached the customer and
> stayed there. Monday you taught me that cancelled orders are not sales. Does your story survive on
> delivered orders? Which branch, which segment, and what would you bet on?"
>
> Meera Raghavan, CEO, Kalpa Retail

> **Kavya's review of the morning.** "Every number you gave Meera this morning was on booked orders.
> Change the definition and rerun every rung. If a branch that held starts to move, find out what it
> is made of before anyone else names it."

This is the exercise twin: every placeholder is filled, every cell has run, and under each step a
line says why the other three options fail. In the exercise twin each placeholder is a lettered choice
in the comment above it; run from the top, and the notebook stops at the first placeholder until you
fill it, which is intended.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
ORDERS = kit.load_records("C2_W01_D02_orders_STUDENT.py")
SEGMENTS = ["Retail-Core", "Retail-Plus", "Business", "Student"]


def tree_for(rows):
    """The revenue tree's leaves for any list of orders, returned as a dictionary (chapter 3)."""
    revenue = 0
    seen = {}
    for order in rows:
        revenue += order["amount"]
        seen[order["customer_id"]] = True
    orders = len(rows)
    customers = len(seen)
    return {"revenue": revenue, "orders": orders, "customers": customers,
            "orders_per_customer": orders / customers if customers else 0,
            "revenue_per_order": revenue / orders if orders else 0}


def pct_change(before, after):
    """Chapter 5's fixed helper: the change every time, in the same type."""
    return round(100 * (after - before) / before, 1)

print(len(ORDERS), "booked orders loaded")

In [ ]:
kit.side_by_side(
    kit.ladder(["Is the delivered drop real?", "The tree on delivered", "Are those lost customers?",
                "Four segments, rolled up", "Mix or rate on delivered"], show=False),
    kit.flow(["booked\n200 orders", "delivered only", "the same ladder", "the sentence"], lit=1, show=False),
)

## Part 1. Is the delivered drop real?

Keep only the orders that reached the customer and stayed, then compare the two closed quarters.

In [ ]:
delivered = []
for order in ORDERS:
    # TODO 1. Which condition keeps the orders that reached the customer and stayed there?
    #   a) order["status"] != "cancelled"
    #   b) order["status"] == "delivered"
    #   c) order["status"] in ("delivered", "returned")
    #   d) order["amount"] > 0
    if __TODO1__:
        delivered.append(order)
dq = {q: [o for o in delivered if o["quarter"] == q] for q in ("Q1", "Q2")}
d1, d2 = tree_for(dq["Q1"]), tree_for(dq["Q2"])
d_change = pct_change(d1["revenue"], d2["revenue"])
kit.columns(["booked", "delivered"], [("Q1", [21000000, d1["revenue"]]), ("Q2", [18700000, d2["revenue"]])],
            fmt=lambda v: f"Rs {v / 1e7:.2f} cr", title="Revenue by quarter on both definitions")
kit.check("138 of the 200 orders were delivered", len(delivered) == 138, f"{len(delivered)}")
kit.check("delivered revenue fell 11.3 percent", d_change == -11.3, f"{d_change}")

**Item 1 in your brief** asks which figure answers Meera. The fall on delivered orders is
Rs 16,40,290, 11.3 percent, on two closed quarters of thirteen weeks: the drop survives the
definition.

## Part 2. The tree on delivered orders

Put rupees on each branch with the bridge in the tree's order: customers first, then orders per
customer, then revenue per order.

In [ ]:
c1, f1, v1 = d1["customers"], d1["orders_per_customer"], d1["revenue_per_order"]
c2, f2, v2 = d2["customers"], d2["orders_per_customer"], d2["revenue_per_order"]
# TODO 2. Which expression is the customers step of the bridge?
#   a) (c2 - c1) * f1 * v1
#   b) c2 * (f2 - f1) * v1
#   c) (c2 - c1) * f2 * v2
#   d) (c2 - c1) / c1
move_customers = __TODO2__
move_frequency = c2 * (f2 - f1) * v1
move_value = c2 * f2 * (v2 - v1)
kit.bridge(("Q1 delivered", d1["revenue"]), [("customers", round(move_customers)), ("orders per customer", round(move_frequency)),
           ("revenue per order", round(move_value))], end_label="Q2 delivered", lit=[0],
           title="The delivered fall along the tree: the customers branch moves now")
kit.check("the bridge lands on Q2 delivered revenue", abs(d1["revenue"] + move_customers + move_frequency + move_value - d2["revenue"]) < 1)
kit.check("customers with a delivered order fell from 54 to 50", (c1, c2) == (54, 50))
kit.check("frequency still carries more rupees than customers", move_frequency < move_customers < 0,
          f"{kit.rupees(round(move_frequency))} against {kit.rupees(round(move_customers))}")

**Item 2 in your brief.** On delivered orders the customers branch takes Rs 10,74,442, frequency
Rs 32,23,327, and revenue per order gives back Rs 26,57,479. Marketing will read the first number as
lost customers. Part 3 finds out what it is.

## Part 3. Are those lost customers?

Four fewer customers had a delivered order in Q2. The morning showed all 69 customers booked in both
quarters. Compare the two views.

In [ ]:
booked_ids = {q: {o["customer_id"] for o in ORDERS if o["quarter"] == q} for q in ("Q1", "Q2")}
deliv_ids = {q: {o["customer_id"] for o in dq[q]} for q in ("Q1", "Q2")}
# TODO 3. Which set holds Q1's delivered customers with no delivered order in Q2 who still booked in Q2?
#   a) deliv_ids["Q1"] - deliv_ids["Q2"]
#   b) booked_ids["Q1"] - booked_ids["Q2"]
#   c) (deliv_ids["Q1"] - deliv_ids["Q2"]) & booked_ids["Q2"]
#   d) deliv_ids["Q2"] - deliv_ids["Q1"]
still_booked = __TODO3__]
gone = deliv_ids["Q1"] - deliv_ids["Q2"]
statuses = {}
for o in ORDERS:
    if o["quarter"] == "Q2" and o["customer_id"] in gone:
        statuses[o["status"]] = statuses.get(o["status"], 0) + 1
kit.bars([("no delivered order in Q2", len(gone)), ("of those, still booked in Q2", len(still_booked)),
          ("lost on booked orders", len(booked_ids["Q1"] - booked_ids["Q2"]))],
         lit=[1], title="The 'lost' delivered customers all ordered again in Q2")
kit.table(["Their Q2 orders ended as", "Orders"], sorted(statuses.items()), caption="What happened to those customers' Q2 orders")
kit.check("19 delivered customers of Q1 had no delivered order in Q2", len(gone) == 19)
kit.check("every one of the 19 booked again in Q2", len(still_booked) == 19)

**Item 3 in your brief.** The customers branch on delivered orders is fulfilment, not acquisition:
the 19 customers who "disappeared" all ordered in Q2, and their orders were cancelled or returned.
The owner of that branch is operations, and the Rs 12 crore still has nothing to replace.

## Part 4. Four segments, rolled up with their weights

Run `tree_for` per segment on delivered orders, check a colleague's roll-up, and count the groups.

In [ ]:
seg1 = {s: tree_for([o for o in dq["Q1"] if o["segment"] == s]) for s in SEGMENTS}
seg2 = {s: tree_for([o for o in dq["Q2"] if o["segment"] == s]) for s in SEGMENTS}
averaged = {q: sum(t["orders_per_customer"] for t in seg.values()) / 4 for q, seg in (("Q1", seg1), ("Q2", seg2))}
# TODO 4. Which roll-up gives the company's delivered orders per customer from the segments?
#   a) the sum of the four segments' orders per customer, divided by 4
#   b) total customers across segments, divided by total orders
#   c) the median of the four segments' orders per customer
#   d) total orders across segments, divided by total customers
def roll_up(seg):
    return __TODO4__
weighted = {"Q1": roll_up(seg1), "Q2": roll_up(seg2)}
changes = {s: pct_change(seg1[s]["orders_per_customer"], seg2[s]["orders_per_customer"]) for s in SEGMENTS}
kit.columns(SEGMENTS, [("Q1", [round(seg1[s]["orders_per_customer"], 2) for s in SEGMENTS]),
                       ("Q2", [round(seg2[s]["orders_per_customer"], 2) for s in SEGMENTS])],
            fmt=lambda v: f"{v:.2f}", lit=[1], title="Delivered orders per customer by segment")
kit.check("the weighted roll-up reproduces 1.50 and 1.14", (round(weighted["Q1"], 2), round(weighted["Q2"], 2)) == (1.5, 1.14))
kit.check("averaging the averages would have said minus 9.2 percent", pct_change(averaged["Q1"], averaged["Q2"]) == -9.2)
# TODO 5. Which test proves that no segment came back without a number?
#   a) None not in changes.values() and len(changes) == 4
#   b) min(changes.values()) < 0
#   c) "Business" in changes
#   d) len(changes) > 0
kit.check("every segment came back with a number", __TODO5__)
kit.check("Retail-Plus fell 42.6 percent per member on delivered orders", changes["Retail-Plus"] == -42.6)

**Item 4 in your brief.** On delivered orders Retail-Plus falls furthest again, 1.85 to 1.06 orders
per member, minus 42.6 percent; Business and Retail-Core fall about 11.5 percent each; Student rises
on two customers. The segment survives the definition.

## Part 5. Mix or rate, on delivered orders

Delivered revenue per order rose 26.0 percent. Split it the way chapter 4 did: price Q2's mix at Q1's
segment rates.

In [ ]:
share1 = {s: seg1[s]["orders"] / d1["orders"] for s in SEGMENTS}
share2 = {s: seg2[s]["orders"] / d2["orders"] for s in SEGMENTS}
at_q2_mix = 0
for s in SEGMENTS:
    # TODO 6. What adds up to revenue per order at Q2's mix and Q1's segment rates?
    #   a) share1[s] * seg2[s]["revenue_per_order"]
    #   b) share2[s] * seg1[s]["revenue_per_order"]
    #   c) share2[s] * seg2[s]["revenue_per_order"]
    #   d) (seg1[s]["revenue_per_order"] + seg2[s]["revenue_per_order"]) / 2
    at_q2_mix += __TODO6__]
mix, rate = at_q2_mix - v1, v2 - at_q2_mix
kit.bridge(("Q1 revenue per order", round(v1)), [("mix", round(mix)), ("rate inside segments", round(rate))],
           end_label="Q2 revenue per order", lo=150000, fmt=kit.rupees, lit=[1],
           title="Delivered revenue per order: here the rate carries more than the mix")
kit.check("mix and rate add back to the whole rise", abs(mix + rate - (v2 - v1)) < 1e-6)
kit.check("on delivered orders the mix explains 44 percent", round(mix / (v2 - v1) * 100) == 44, f"{mix / (v2 - v1) * 100:.1f}")

**Item 5 in your brief.** On booked orders the mix explained 69 percent of the rise; on delivered
orders 44 percent, because Business's delivered orders grew in size, from Rs 10,24,651 to
Rs 11,60,405 each on average. The consumer segments still paid about the same. The definition moved
the split; it did not create a price signal in the consumer business.

## The sentence

In [ ]:
# TODO 7. Which segment leads the sentence as the behaviour finding on delivered orders?
#   a) "Business"
#   b) "Student"
#   c) "all four segments"
#   d) "Retail-Plus"
lead = __TODO7__
kit.flow([f"drop real\n{d_change}% delivered", f"branch\nfrequency {kit.rupees(round(move_frequency))}",
          f"customers branch\n19 still booked", f"segment\n{lead} {changes[lead]}%", "two hypotheses\nand their evidence"],
         kinds=["known", "bad", "unknown", "bad", "known"], title="The sentence to Meera, on delivered orders")
kit.check("the lead segment is the largest per-member fall", lead == min(changes, key=changes.get))

In [ ]:
kit.check_summary()
print("Answer string for the TODOs: b a c d a b d.")